In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# OpenAI Assistants API function calling 기능 실습 (2025년 3월 기준)

본 실습에서는 OpenAI Assistants API의 **함수 호출(Function Calling)** 기능을 사용하여, AI 어시스턴트가 외부 함수를 호출하고 그 결과를 응답에 활용하는 방법을 단계별로 살펴보겠습니다. 2025년 3월 기준의 OpenAI 공식 API 문서를 바탕으로 최신 함수 호출 기능과 모범 사례를 반영하여 진행합니다. 이 실습은 기존에 ChatGPT API 등을 사용해본 경험이 있고, Assistants API의 새로운 함수 호출 기능에 관심이 있는 개발자를 대상으로 합니다.

## 1. 함수 호출 기능 소개

함수 호출 기능은 모델이 사전에 정의된 함수를 필요에 따라 실행하도록 도와주는 강력한 도구입니다. 이 기능을 통해 어시스턴트는 자체 지식에 없는 **실시간 정보 조회**나 **복잡한 계산 작업** 등을 외부 함수에 위임할 수 있습니다. 예를 들어 사용자가 날씨를 물어보면, 어시스턴트는 날씨 API와 연결된 함수를 호출하여 최신 기온을 가져온 뒤 응답할 수 있습니다. 이처럼 함수 호출을 활용하면:
- **실시간/외부 데이터 활용:** 모델이 최신 정보(날씨, 주가, 뉴스 등)를 함수로부터 받아와 답변에 반영할 수 있습니다.
- **모델 한계 보완:** 수학 계산, 데이터베이스 질의 등 모델이 직접 처리하기 어려운 요청을 외부 로직으로 해결할 수 있습니다.
- **도메인 확장:** 개발자가 정의한 임의의 함수로 모델의 기능을 확장할 수 있어, 특정 분야나 서비스에 특화된 어시스턴트를 구현할 수 있습니다.

OpenAI의 GPT-3.5/4 모델 중 2023년 하반기 이후 출시된 버전들은 이 함수 호출 기능을 지원합니다. 모델에게 함수 목록을 제공하면, 질문 의도에 따라 적절한 함수를 선택해 필요한 인수를 함께 호출 형식으로 응답을 반환합니다. 이제 이러한 함수 호출을 구현하는 방법을 예제로 알아보겠습니다.

## 2. API 키 설정 및 OpenAI 클라이언트 초기화

OpenAI API를 사용하기 위해 먼저 API 키를 준비해야 합니다. API 키는 OpenAI 계정의 대시보드에서 생성할 수 있으며, 노출되지 않도록 환경 변수 등에 저장하여 사용합니다. 본 예제에서는 **python-dotenv**를 이용해 `.env` 파일에 저장된 키를 로드하고, OpenAI Python SDK의 `OpenAI` 클라이언트를 초기화합니다. 
아래 코드에서는 `.env` 파일에서 키를 읽어와 `client = OpenAI()`로 클라이언트를 생성합니다. (API 키가 올바르게 설정되어 있으면 `OpenAI()` 생성자에서 자동으로 키를 불러옵니다.)


In [24]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

## 3. 외부 함수 정의 및 테스트

이제 함수 호출 기능을 체험하기 위한 예시로 **날씨 정보를 가져오는 함수**를 만들어보겠습니다. 사용자가 "서울 날씨 어때요?"라고 물어보면, 어시스턴트가 이 함수를 호출하여 실시간 날씨 정보를 얻어 답변하도록 해볼 것입니다. 

예를 위해 간단한 `get_weather` 함수를 정의하겠습니다. 이 함수는 위도(latitude)와 경도(longitude)를 받아 해당 위치의 현재 기온을 섭씨로 반환합니다. 구현에는 오픈 메테오(Open-Meteo)라는 공개 기상 API를 사용하여, 주어진 좌표의 현재 기온 데이터를 가져옵니다. 함수는 `requests` 라이브러리를 통해 API를 호출하고, 응답 JSON에서 온도 값을 추출하여 반환합니다. 

함수를 정의한 후, 예시 좌표에 대해 함수를 호출해 제대로 동작하는지 확인해 보겠습니다. 서울의 대략적인 좌표(위도 37.484859, 경도 126.930086)를 입력으로 주었을 때 온도가 잘 반환되는지 출력해 보겠습니다.



In [25]:
# 위도/경도를 입력하면 섭씨 온도를 return
import requests
def get_weather(latitude=37.484859, longitude=126.930086):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    # print(url)
    response = requests.get(url)
    data = response.json().get('current')
    if data:
        return data.get('temperature_2m')
    else:
        return 'error'
get_weather()

26.4

In [15]:
get_weather(133,126)

'error'

위 코드에서는 client.beta.assistants.create 메서드를 사용해 새로운 어시스턴트를 생성했습니다. 주요 파라미터를 살펴보면:

- name: 어시스턴트의 이름을 지정합니다. 콘솔이나 리스트에서 구분하기 위한 용도이며, 모델 응답 내용에는 영향을 미치지 않습니다.

- instructions: 시스템 레벨의 지시어로, 해당 어시스턴트가 모든 대화에서 따르게 될 기본 규칙이나 역할을 정의합니다. 여기서는 "친절한 도움말 어시스턴트"라는 성격을 부여했습니다. (이 지시어는 기존 ChatGPT의 시스템 메시지와 유사한 역할입니다.)

- model: 어시스턴트가 사용할 언어 모델을 지정합니다. 최신 기능을 쓰려면 OpenAI가 Nov 2023 이후 출시한 모델(-1106가 붙은 모델명)을 권장합니다. 

- assistant 생성을 제공하는 모델
    GPT-4 계열:
    gpt-4 - 기본 GPT-4 모델
    gpt-4-turbo - 더 빠르고 효율적인 GPT-4
    gpt-4o - 최신 멀티모달 모델
    gpt-4o-mini - 경량화된 GPT-4o

    GPT-3.5 계열:
    gpt-3.5-turbo - 빠르고 효율적인 모델(추천)

    기타:
    dall-e-3 - 이미지 생성용
    whisper-1 - 음성 인식용
    tts-1, tts-1-hd - 텍스트 음성 변환용


- tools: 어시스턴트에 활성화할 도구 목록입니다. 기본적인 Q&A 챗봇에서는 특별한 툴이 필요 없으므로 비워두었습니다. (tools=[] 또는 생략) 나중에 코드 인터프리터나 함수 호출 등을 사용하게 될 경우 이 필드를 설정합니다.

어시스턴트가 성공적으로 생성되면 고유 ID (assistant.id)가 반환됩니다. 이 ID는 이후 대화 스레드에서 어떤 어시스턴트를 사용할지 지정할 때 필요하므로 저장해둡니다. (참고로, client.beta.assistants.list()를 호출하면 계정 내 모든 어시스턴트 목록을 확인할 수 있습니다.) 

이제 이 어시스턴트와 대화를 시작해보겠습니다. 대화를 하기 위해서는 **스레드(Thread)**를 생성해야 합니다. 스레드는 개별적인 대화 세션을 나타내며, 사용자와 어시스턴트 간의 메시지(Message) 목록을 포함합니다. 하나의 어시스턴트는 여러 개의 스레드를 가질 수 있는데, 예를 들어 같은 어시스턴트를 여러 사용자가 동시에 이용하면 각 사용자별로 별도 스레드가 생깁니다. 

스레드를 만들고, 그 안에 사용자 메시지를 추가한 뒤, 어시스턴트의 답변을 요청하는 일련의 과정을 코드로 수행해봅시다:


In [26]:
tools=[{
    'type':'function',
    'function':{
        'name':'get_weather',
        'description':'저장된 좌표의 현재 온도를 섭씨 단위로 return 합니다',
        'parameters':{
            'type':'object',
            'properties' :{
                'latitude':{'type':'number'},
                'longitude':{'type':'number'}
            },
            'required': ['latitude', 'longitude'],  # 반드시 입력 요구 파라미터
            'additionalProperties':False # 지정된 properties 외에는 추가 허용 안함

        } # parameters
    } # function
}] # tool
tools

[{'type': 'function',
  'function': {'name': 'get_weather',
   'description': '저장된 좌표의 현재 온도를 섭씨 단위로 return 합니다',
   'parameters': {'type': 'object',
    'properties': {'latitude': {'type': 'number'},
     'longitude': {'type': 'number'}},
    'required': ['latitude', 'longitude'],
    'additionalProperties': False}}}]

In [27]:
messages = [{'role':'user', 'content': '오늘 서울 날씨 어때요?'}]
completion = client.chat.completions.create(
#     model='gpt-4.1-nano',
#     messages=messages,
#     tools=tools
)
completion

ChatCompletion(id='chatcmpl-Bo3ZPkHIrafx0sGiCIuc2srcAA7rd', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_7TLNlFbwPw5TkqSJoY0NEqIg', function=Function(arguments='{"latitude":37.5665,"longitude":126.978}', name='get_weather'), type='function')]))], created=1751269823, model='gpt-4.1-nano-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_38343a2f8f', usage=CompletionUsage(completion_tokens=23, prompt_tokens=66, total_tokens=89, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [34]:
completion.choices[0].message.tool_calls

[ChatCompletionMessageToolCall(id='call_7TLNlFbwPw5TkqSJoY0NEqIg', function=Function(arguments='{"latitude":37.5665,"longitude":126.978}', name='get_weather'), type='function')]

In [42]:
import json
for tool_call in completion.choices[0].message.tool_calls:
    fun_name = tool_call.function.name
    arg = tool_call.function.arguments  # <class 'str'> {"latitude":37.5665,"longitude":126.978}
    args = json.loads(arg)   # 스트링 타입을 제이슨으로 변경
    if fun_name == 'get_weather':
        result = get_weather(args['latitude'], args['longitude'])
        print(type(result), result)

<class 'float'> 26.5


In [43]:
print(messages)

[{'role': 'user', 'content': '오늘 서울 날씨 어때요?'}]


In [ ]:
# llm 모델이 반환한 메세지 객체
completion.choices[0].message

In [44]:
messages.append(completion.choices[0].message)
messages.append({
    'role':'tool',
    'tool_call_id':tool_call.id,
    'content':str(result)
})
print(messages)

[{'role': 'user', 'content': '오늘 서울 날씨 어때요?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_7TLNlFbwPw5TkqSJoY0NEqIg', function=Function(arguments='{"latitude":37.5665,"longitude":126.978}', name='get_weather'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_7TLNlFbwPw5TkqSJoY0NEqIg', 'content': '26.5'}]


In [45]:
completion2 = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages,
    tools=tools
)
completion2

ChatCompletion(id='chatcmpl-Bo43asoX3Rm1zPjM5bsHD4Ns6gnJO', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='오늘 서울의 날씨는 약 26.5도입니다. 날씨가 따뜻하니 외출 시 참고하세요!', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1751271694, model='gpt-4.1-nano-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_38343a2f8f', usage=CompletionUsage(completion_tokens=28, prompt_tokens=100, total_tokens=128, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [46]:
completion2.choices[0].message.content

'오늘 서울의 날씨는 약 26.5도입니다. 날씨가 따뜻하니 외출 시 참고하세요!'